# 01 - Primeiros passos com Python para RHNeste notebook vamos sair do Python basico e entrar em tabelas reais. O objetivo e aprender a abrir um CSV, entender linhas e colunas, filtrar dados, agrupar informacoes e criar indicadores simples.Tudo sera comentado como se fosse a primeira vez. Leia os comentarios do codigo com atencao: eles sao parte da aula.

## 1. Importar bibliotecas e localizar a pasta de dadosAntes de analisar dados, precisamos carregar ferramentas. Aqui usamos `pathlib` para lidar com caminhos de arquivos e `pandas` para trabalhar com tabelas.

In [ ]:
# pathlib ajuda a montar caminhos de arquivo de forma mais segura.from pathlib import Path# pandas e a biblioteca principal para trabalhar com tabelas.import pandas as pd# Esta configuracao pede ao pandas para mostrar mais colunas quando exibimos tabelas.pd.set_option("display.max_columns", 50)# Path.cwd() mostra a pasta atual em que o notebook esta rodando.# Se o notebook estiver rodando de dentro da pasta notebooks, subimos uma pasta.BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()# A pasta dados fica dentro da raiz do projeto.DADOS = BASE_DIR / "dados"# Mostramos o caminho para confirmar que o Python encontrou a pasta certa.DADOS

## 2. Abrir uma planilha CSVCSV e um arquivo de texto com colunas separadas por virgula. Ele funciona como uma planilha simples.Vamos abrir a base de vagas de recrutamento. Cada linha representa uma vaga.

In [ ]:
# pd.read_csv() le um arquivo CSV e devolve uma tabela chamada DataFrame.# Guardamos essa tabela na variavel vagas.vagas = pd.read_csv(DADOS / "vagas_recrutamento.csv")# head() mostra as primeiras 5 linhas.# Isso ajuda a conferir se o arquivo abriu corretamente.vagas.head()

## 3. Entender tamanho, colunas e tiposAntes de calcular qualquer indicador, fazemos uma inspecao inicial. Esta etapa evita erros simples, como tentar calcular media de uma coluna de texto.

In [ ]:
# shape devolve uma dupla: quantidade de linhas e quantidade de colunas.print(f"Linhas: {vagas.shape[0]}")print(f"Colunas: {vagas.shape[1]}")# columns mostra os nomes das colunas.print("Colunas da base:")print(list(vagas.columns))# dtypes mostra o tipo de cada coluna.# int64 e numero inteiro; object geralmente e texto; float64 e numero decimal.vagas.dtypes

## 4. Selecionar colunasSelecionar colunas e como escolher quais campos queremos ver na planilha. Isso ajuda a focar a analise.

In [ ]:
# Criamos uma lista com os nomes das colunas que queremos visualizar.colunas_para_ver = ["vaga_id", "cargo", "area", "source_of_hire", "time_to_hire_dias"]# Usamos a lista dentro de vagas[...] para mostrar apenas essas colunas.vagas[colunas_para_ver].head()

## 5. Filtrar linhasFiltro e uma das operacoes mais importantes. Em vez de olhar todas as vagas, podemos olhar apenas uma area, uma fonte ou uma senioridade.

In [ ]:
# Esta condicao verifica, linha por linha, se a area e igual a RH.condicao_rh = vagas["area"] == "RH"# Quando usamos a condicao dentro de vagas[...], ficamos apenas com as linhas True.vagas_rh = vagas[condicao_rh]# Mostramos algumas colunas para conferir o filtro.vagas_rh[["vaga_id", "cargo", "area", "source_of_hire", "time_to_hire_dias"]]

## 6. Criar indicadores como novas colunasNo Excel, criaríamos uma nova coluna com formula. Em pandas, fazemos o mesmo escrevendo a formula em codigo.

In [ ]:
# Criamos uma copia para evitar alterar a tabela original sem querer.vagas_com_indicadores = vagas.copy()# Custo por contratado = custo de divulgacao dividido pela quantidade de contratados.# Este indicador ajuda a comparar eficiencia de fontes de contratacao.vagas_com_indicadores["custo_por_contratado"] = (    vagas_com_indicadores["custo_divulgacao"] / vagas_com_indicadores["contratados"])# Candidatos por entrevista mostra quantas candidaturas geram uma entrevista, em media, por vaga.vagas_com_indicadores["candidatos_por_entrevista"] = (    vagas_com_indicadores["candidatos"] / vagas_com_indicadores["entrevistas"])vagas_com_indicadores[["vaga_id", "source_of_hire", "custo_por_contratado", "candidatos_por_entrevista"]].head()

## 7. Agrupar dados: a tabela dinamica do pandas`groupby` agrupa linhas por uma categoria. Depois usamos `agg` para calcular contagem, media, soma ou outros resumos.

In [ ]:
# Agrupamos por source_of_hire porque queremos comparar fontes de contratacao.resumo_source = (    vagas_com_indicadores    .groupby("source_of_hire")    .agg(        # Contamos quantas vagas existem em cada fonte.        vagas=("vaga_id", "count"),        # Calculamos o time to hire medio.        time_to_hire_medio=("time_to_hire_dias", "mean"),        # Calculamos candidatos medios por vaga.        candidatos_medios=("candidatos", "mean"),        # Calculamos o custo medio de divulgacao.        custo_medio=("custo_divulgacao", "mean"),        # Calculamos o custo medio por contratado.        custo_por_contratado_medio=("custo_por_contratado", "mean"),    )    # Arredondamos para facilitar leitura.    .round(1)    # Ordenamos pela menor media de time to hire.    .sort_values("time_to_hire_medio"))resumo_source

## 8. Ordenar e interpretarOrdenar ajuda a ver extremos: maior, menor, mais caro, mais rapido. Mas lembre: ordenar nao explica causa, apenas organiza evidencias.

In [ ]:
# Selecionamos as 5 vagas com maior time to hire.# Isso ajuda a investigar casos mais demorados.vagas_mais_lentas = vagas_com_indicadores.sort_values("time_to_hire_dias", ascending=False).head(5)vagas_mais_lentas[["vaga_id", "cargo", "area", "senioridade", "source_of_hire", "time_to_hire_dias"]]

## 9. Exercicios1. Troque o filtro de `RH` para `TI`.2. Calcule o time to hire medio por `recrutador`.3. Descubra qual area teve mais candidatos por vaga.4. Crie uma coluna chamada `entrevistas_por_oferta`.5. Escreva uma conclusao com: evidencia, interpretacao e limitacao.

In [ ]:
# Espaco para exercicios.# Dica: copie uma celula anterior, altere uma parte pequena e execute.